In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
dfield_approach_elastix_path = os.path.abspath(os.path.join(current_path, '../'))
sys.path.append(base_path)
sys.path.append(dfield_approach_elastix_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display


import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc

import constants as cts
import dfield_applier as dfield_applier
import utils.util_freesurfer_segmentation as util_freesurfer

np.set_printoptions(precision=4)



In [2]:
def visualize_patch(patch_size=32, simple=True):
    %matplotlib widget
    mask_target_path_name = pil.get_segmentation_path_name(3, "00039", simple=simple, absolute=True)
    mask_target, _ = nfc.load_nifti(mask_target_path_name)
    mask_target = util_freesurfer.seg_to_rgbseg(mask_target)

    pinit = 100
    pend = pinit + patch_size
    mask_target[pinit:pend, pinit:pend, pinit:pend] = 0
    fc.imgshow_3D("", mask_target)

# visualize_patch(16, simple=False)
# visualize_patch(16, simple=True)

In [3]:

def compute_tissue_weigthed_dice(patch_target, patch_data):
    tissue_in_target, nb_tissue_voxels_in_target = np.unique(patch_target, return_counts=True)
    tissue_in_data, nb_tissue_voxels_in_data = np.unique(patch_target, return_counts=True)

    tissue_volume_in_target = nb_tissue_voxels_in_target / np.prod(patch_target.shape)
    tissue_volume_in_data = nb_tissue_voxels_in_data / np.prod(patch_data.shape)

    weighted_sum = 0
    total_tissue_weight = 0
    for i, tissue_index in enumerate(tissue_in_target):
        if tissue_index in tissue_in_data:
            tissue_weight_target = tissue_volume_in_target[i]

            i_data = np.where(tissue_in_data == tissue_index)[0][0]
            tissue_weight_data = tissue_volume_in_data[i_data]

            mask1 = patch_target == tissue_index
            mask2 = patch_data == tissue_index

            tissue_dice = util.compute_dice_coefficient(mask1, mask2)
            # weighted_sum += (tissue_weight_target + tissue_weight_data) * tissue_dice
            # tissue weigtehd version
            tissue_weight = 0
            # if tissue_index == 4 or tissue_index == 43:
            #     tissue_weight = 10
            # elif tissue_index == 2 or tissue_index == 41:
            #     tissue_weight = 0.01
            weighted_sum += (tissue_weight + tissue_weight_target + tissue_weight_data) * tissue_dice
            total_tissue_weight += tissue_weight

    dice = weighted_sum / (2+total_tissue_weight)
    # print("Dice: ", dice)
    return dice



In [4]:
%matplotlib widget

def sort_array_by_columns(array):
    """
    Sorts each column (3rd dimension) individually of a 3-dimensional array.
    """
    ## Slow way to sort the array by columns
    # array_sorted = np.zeros(array.shape)
    # for patch_index in range(len(array)):
    #     sorted_indices = np.argsort(array[:, patch_index, 1])[::-1]
    #     array_sorted[:, patch_index, :] = array[sorted_indices, patch_index, :]

    ## Fats way to sort the array by columns
    sorted_indices = np.argsort(array[:, :, 1], axis=0)[::-1]
    array_sorted = array[sorted_indices, np.arange(array.shape[1]), :]

    return array_sorted

def compute_tissue_similarity_table(mask_target, tissue_ids, dataset_ids, simple_mask=False):
    nb_max_data_images = len(dataset_ids)
    # tissue_indexs = np.unique(mask_target)

    similarity_tissue_table = np.zeros((nb_max_data_images, len(tissue_ids), 2))

    img_index = 0
    print("Comuting dice for each tissue of each image")
    for id_data in tqdm(dataset_ids):
        mask_data, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_data, simple=simple_mask, absolute=True))
        similarity_tissue_table[img_index, :, 0] = img_index
        similarity_tissue_table[img_index, :, 1] = util.compute_multiple_dice_coefficients(mask_target, mask_data, return_dict=False, seg_indexs=tissue_ids)

        img_index += 1
    return similarity_tissue_table


In [5]:
def compute_tissue_based_dfield(similarity_tissue_table, tissue_ids, dataset_ids, img_size, nb_best_images=10, simple_mask=False):
    # 1. sort patches_dice by columns individually to obtain the best image for each patch
    similarity_tissue_table = sort_array_by_columns(similarity_tissue_table)
    most_similar_tissues = similarity_tissue_table[:nb_best_images]

    # 2. compute best patches weights (l1 normalized dice scores)
    l1_norm = np.linalg.norm(most_similar_tissues[:, :, 1], ord=1, axis=0)
    l1_norm[l1_norm == 0] = 1
    weights_columns = most_similar_tissues[:,:,1] / l1_norm
    most_similar_tissues = np.concatenate((most_similar_tissues, weights_columns[:, :, np.newaxis]), axis=2)

    # 3. ontain dfields index to load
    dfields_data_to_load = np.unique(most_similar_tissues[:, :, 0])
    dfields_data_to_load = [int(i) for i in dfields_data_to_load if i >= 0]
    # print("dfields_data_to_load", [target_ids[i] for i in dfields_data_to_load])

    # # print("most_similar_tissues")
    # most_similar_tissues = np.sum(most_similar_tissues[:,:,1], axis=0)
    # print("Min sum of dice scores for the best patches")
    # for i in range(nb_best_images):
    #     print(most_similar_tissues[i, np.argmin(most_similar_tissues), :])
    
    # print("Max sum of dice scores for the best patches")
    # for i in range(nb_best_images):
    #     print(most_similar_tissues[i, np.argmax(most_similar_tissues), :])
        
    # 4. compute dfield patch weighted mean
    dfield_constructed = np.zeros((*img_size, 3))

    affine = None
    # 5. recontruct the dfield from the patches
    # looping over the best dfields in order to be faster
    print("constructing target dfield from the best dfields")
    dfield_data_masked = np.zeros((*img_size, 3))
    for img_data_index in tqdm(dfields_data_to_load):
        id_data = dataset_ids[img_data_index]
        dfield_data_path_name = cts.get_image_path_name(id_data, cts.PFILE_0038, cts.INAME_DFIELD, subfolder=cts.FOLDER_REGISTRATIONS)
        dfield_data, affine = nfc.load_nifti(dfield_data_path_name)
        dfield_data = dfield_data[:,:,:,0]
        
        mask_data, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_data, simple=simple_mask, absolute=True))


        # ------ Low version 
        # # looping over the tissues
        # for tissue_index in range(most_similar_tissues.shape[1]):
        #     current_patch_table = most_similar_tissues[:, tissue_index, :]
        #     # looping over the images to find the current data image
        #     for img_index, img_dice, img_weight  in current_patch_table:
        #         # check the loaded best dfield is part of the weigthted mean of the current tissue
        #         if img_index == img_data_index:
        #             tissue_mask = mask_data == tissue_ids[tissue_index]
        #             dfield_data_masked *= 0
        #             dfield_data_masked[tissue_mask] = dfield_data[tissue_mask]
                    
        #             # add the dfield to the weighted mean of the current tissue
        #             dfield_constructed += img_weight * dfield_data_masked
 


        # ------ Fast version
        for tissue_index in range(most_similar_tissues.shape[1]):
            current_patch_table = most_similar_tissues[:, tissue_index, :]
            img_indices = current_patch_table[:, 0]
            img_weights = current_patch_table[:, 2]

            matching_indices = np.where(img_indices == img_data_index)[0]

            for index in matching_indices:
                tissue_mask = mask_data == tissue_ids[tissue_index]
                dfield_data_masked *= 0
                dfield_data_masked[tissue_mask] = dfield_data[tissue_mask]

                dfield_constructed += img_weights[index] * dfield_data_masked

    dfield_constructed = np.expand_dims(dfield_constructed, axis=-2)
    return dfield_constructed, affine




def compute_tissue_based_dfield2(similarity_tissue_table, tissue_ids, mask_target, dataset_ids, img_size, nb_best_images=10, simple_mask=False):
    # 1. sort patches_dice by columns individually to obtain the best image for each patch
    similarity_tissue_table = sort_array_by_columns(similarity_tissue_table)
    most_similar_tissues = similarity_tissue_table[:nb_best_images]

    # 2. compute best patches weights (l1 normalized dice scores)
    l1_norm = np.linalg.norm(most_similar_tissues[:, :, 1], ord=1, axis=0)
    l1_norm[l1_norm == 0] = 1
    weights_columns = most_similar_tissues[:,:,1] / l1_norm
    most_similar_tissues = np.concatenate((most_similar_tissues, weights_columns[:, :, np.newaxis]), axis=2)

    # 3. ontain dfields index to load
    dfields_data_to_load = np.unique(most_similar_tissues[:, :, 0])
    dfields_data_to_load = [int(i) for i in dfields_data_to_load if i >= 0]

    # 4. compute dfield patch weighted mean
    dfield_constructed = np.zeros((*img_size, 3))

    affine = None
    # 5. recontruct the dfield from the patches
    # looping over the best dfields in order to be faster
    print("constructing target dfield from the best dfields")
    dfield_data_masked = np.zeros((*img_size, 3))
    for img_data_index in tqdm(dfields_data_to_load):
        id_data = dataset_ids[img_data_index]
        dfield_data_path_name = cts.get_image_path_name(id_data, cts.PFILE_0038, cts.INAME_DFIELD, subfolder=cts.FOLDER_REGISTRATIONS)
        dfield_data, affine = nfc.load_nifti(dfield_data_path_name)
        dfield_data = dfield_data[:,:,:,0]
        
        # mask_data, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_data, simple=simple_mask, absolute=True))

        for tissue_index in range(most_similar_tissues.shape[1]):
            current_patch_table = most_similar_tissues[:, tissue_index, :]
            img_indices = current_patch_table[:, 0]
            img_weights = current_patch_table[:, 2]

            matching_indices = np.where(img_indices == img_data_index)[0]

            for index in matching_indices:
                tissue_mask = mask_target == tissue_ids[tissue_index]
                dfield_data_masked *= 0
                dfield_data_masked[tissue_mask] = dfield_data[tissue_mask]

                dfield_constructed += img_weights[index] * dfield_data_masked

    dfield_constructed = np.expand_dims(dfield_constructed, axis=-2)
    return dfield_constructed, affine

### Create patch based dfields

In [6]:

def create_patch_base_dfield(id_target, matchin_ids, nb_best_images_list, nb_used_images=300):
    dataset_ids = matchin_ids[:nb_used_images]
    simple_segmentation = False
    
    mask_target, _ = nfc.load_nifti(pil.get_segmentation_path_name(3, id_target, simple=simple_segmentation, absolute=True))
    tissue_ids = np.unique(mask_target)
    similarity_patches_table = compute_tissue_similarity_table(mask_target, tissue_ids, dataset_ids, simple_mask=simple_segmentation)
    for nb_best_images in nb_best_images_list:
        # dfield_target, affine = compute_tissue_based_dfield(similarity_patches_table, tissue_ids, dataset_ids, mask_target.shape, nb_best_images=nb_best_images, simple_mask=simple_segmentation)
        dfield_target, affine = compute_tissue_based_dfield2(similarity_patches_table, tissue_ids, mask_target, dataset_ids, mask_target.shape, nb_best_images=nb_best_images, simple_mask=simple_segmentation)
        nfc.save_nifti(dfield_target, affine, cts.get_tissue_based_dfield_path_name(id_target, f"MT{len(tissue_ids)}", cts.PFILE_0038, nb_best_images))



matchin_ids = pil.get_matching_ids()
# id_indexs = [301,302,303,304,305,306,307]
id_indexs = [302]


for id_index in id_indexs:
    id_target = matchin_ids[id_index]
    create_patch_base_dfield(id_target, matchin_ids, [10, 100], nb_used_images=300)
    # create_patch_base_dfield(id_target, matchin_ids, [1], nb_used_images=2)


Comuting dice for each tissue of each image


100%|██████████| 300/300 [10:33<00:00,  2.11s/it]


constructing target dfield from the best dfields


100%|██████████| 151/151 [03:30<00:00,  1.39s/it]


constructing target dfield from the best dfields


100%|██████████| 291/291 [24:07<00:00,  4.97s/it]
